# 1. IMPORT LIBRARIES


In [ ]:
!pip install optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score, make_scorer, classification_report
import lightgbm as lgb
import optuna
import warnings
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 15.3 MB/s eta 0:00:00


# 2. LOAD DATA

In [ ]:
!unzip super-ai-engineer-ss-6-heart-disease-prediction

Archive:  super-ai-engineer-ss-6-heart-disease-prediction.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [ ]:
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
sub_df   = pd.read_csv('sample_submission.csv')

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nColumns:", train_df.columns.tolist())

# The column name was 'History of HeartDisease or Attack'
target_col = 'History of HeartDisease or Attack'
if target_col in train_df.columns:
    print(f"\nTarget distribution ({target_col}):\n", train_df[target_col].value_counts(normalize=True))
else:
    print(f"\nColumn '{target_col}' not found. Please check the printed columns above.")

Train shape: (223084, 20)
Test shape : (74361, 19)

Columns: ['ID', 'History of HeartDisease or Attack', 'High Blood Pressure', 'Told High Cholesterol', 'Cholesterol Checked', 'Body Mass Index', 'Smoked 100+ Cigarettes', 'Diagnosed Stroke', 'Diagnosed Diabetes', 'Leisure Physical Activity', 'Heavy Alcohol Consumption', 'Health Care Coverage', 'Doctor Visit Cost Barrier', 'General Health', 'Difficulty Walking', 'Sex', 'Education Level', 'Income Level', 'Age', 'Vegetable or Fruit Intake (1+ per Day)']

Target distribution (History of HeartDisease or Attack):
 History of HeartDisease or Attack
No     0.918388
Yes    0.081612
Name: proportion, dtype: float64


# 3. CLEAN DATA

In [ ]:
# ดู unique จริงๆ ว่าต่างจาก map ยังไง
for col in ['General Health', 'Education Level', 'Income Level']:
    print(f"\n--- {col} ---")
    print(repr(train_df[col].unique()))
    # repr() จะเห็น whitespace/hidden chars ชัดขึ้น


--- General Health ---
array(['Very Poor', 'Fair', 'Good', 'Poor', 'Excellent', nan],
      dtype=object)

--- Education Level ---
array(['High school graduate', 'College graduate', 'Some high school',
       'Some college or technical school', 'Elementary',
       'Never attended school'], dtype=object)

--- Income Level ---
array(['$15,000 to less than $20,000', 'Less than $10,000',
       '$75,000 or more', '$35,000 to less than $50,000',
       '$20,000 to less than $25,000', '($10,000 to less than $15,000',
       '$50,000 to less than $75,000', '$25,000 to less than $35,000'],
      dtype=object)


In [ ]:
def clean_data(df):
    df = df.copy()
    ids = df['ID'].copy() if 'ID' in df.columns else None
    if 'ID' in df.columns:
        df = df.drop('ID', axis=1)

    binary_cols = [
        'High Blood Pressure', 'Told High Cholesterol', 'Cholesterol Checked',
        'Smoked 100+ Cigarettes', 'Diagnosed Stroke', 'Diagnosed Diabetes',
        'Leisure Physical Activity', 'Heavy Alcohol Consumption',
        'Health Care Coverage', 'Doctor Visit Cost Barrier',
        'Difficulty Walking', 'Vegetable or Fruit Intake (1+ per Day)'
    ]
    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().map({'Yes': 1, 'No': 0})

    df['Sex'] = df['Sex'].astype(str).str.strip().map({'Male': 1, 'Female': 0})

    health_map = {'Very Poor': 0, 'Poor': 1, 'Fair': 2, 'Good': 3, 'Very good': 4, 'Excellent': 5}
    df['General Health'] = df['General Health'].astype(str).str.strip().map(health_map)

    edu_map = {
        'Never attended school': 0, 'Elementary': 1,
        'Some high school': 2, 'High school graduate': 3,
        'Some college or technical school': 4, 'College graduate': 5
    }
    df['Education Level'] = df['Education Level'].astype(str).str.strip().map(edu_map)

    df['Income Level'] = df['Income Level'].astype(str).str.strip().str.replace(r'^\(', '', regex=True)
    income_map = {
        'Less than $10,000': 0, '$10,000 to less than $15,000': 1,
        '$15,000 to less than $20,000': 2, '$20,000 to less than $25,000': 3,
        '$25,000 to less than $35,000': 4, '$35,000 to less than $50,000': 5,
        '$50,000 to less than $75,000': 6, '$75,000 or more': 7
    }
    df['Income Level'] = df['Income Level'].map(income_map)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df = df.fillna(df.median(numeric_only=True))

    remaining = df.isnull().sum().sum()
    print("ไม่มี NaN" if remaining == 0 else f"NaN เหลือ {remaining}")
    return df, ids

# 4. PREPROCESS

In [ ]:
# 4. PREPROCESS
TARGET = 'History of HeartDisease or Attack'
train_clean, _   = clean_data(train_df)
test_clean,  ids = clean_data(test_df)

X = train_clean.drop(TARGET, axis=1)

# Map Target Yes/No → 1/0
y = train_clean[TARGET].map({'Yes': 1, 'No': 0})

# Drop rows ที่ Target เป็น NaN (ถ้ามี)
nan_mask = y.isna()
print(f"Target NaN: {nan_mask.sum()} rows → drop ออก")
X = X[~nan_mask]
y = y[~nan_mask]

print(f"\nX shape: {X.shape}")
print(f"y distribution:\n{y.value_counts(normalize=True)}")
print(f"\nClass imbalance → 0: {(y==0).sum()}, 1: {(y==1).sum()}")

neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

NaN เหลือ 1694
ไม่มี NaN
Target NaN: 1694 rows → drop ออก

X shape: (221390, 18)
y distribution:
History of HeartDisease or Attack
0.0    0.918388
1.0    0.081612
Name: proportion, dtype: float64

Class imbalance → 0: 203322, 1: 18068
scale_pos_weight = 11.25


In [ ]:
#4.5: FEATURE ENGINEERING
def add_features(df):
    df = df.copy()

    # Metabolic Syndrome Proxy (0-4 แต้ม)
    df['Metabolic_Score'] = (
        df['High Blood Pressure'].astype(int) +
        df['Told High Cholesterol'].astype(int) +
        (df['Body Mass Index'] > 30).astype(int) +
        df['Diagnosed Diabetes'].astype(int)
    )

    # Age × Risk Behaviors
    df['Age_x_Smoking'] = df['Age'] * df['Smoked 100+ Cigarettes']
    df['Age_x_Alcohol'] = df['Age'] * df['Heavy Alcohol Consumption']
    df['Age_x_BP']      = df['Age'] * df['High Blood Pressure']

    # Healthy Lifestyle Index
    df['Lifestyle_Score'] = (
        df['Vegetable or Fruit Intake (1+ per Day)'].astype(int) +
        df['Leisure Physical Activity'].astype(int) -
        df['Heavy Alcohol Consumption'].astype(int) -
        df['Smoked 100+ Cigarettes'].astype(int)
    )

    # Healthcare Access
    df['No_Healthcare'] = (
        (df['Health Care Coverage'] == 0) |
        (df['Doctor Visit Cost Barrier'] == 1)
    ).astype(int)

    return df

# ต่อจาก Cell 4 ที่ define X, y แล้ว
X = add_features(X)
test_clean_fe = add_features(test_clean)

print(f"Features: 18 → {X.shape[1]} columns")
print(f"New features: {[c for c in X.columns if c not in train_clean.columns]}")

Features: 18 → 24 columns
New features: ['Metabolic_Score', 'Age_x_Smoking', 'Age_x_Alcohol', 'Age_x_BP', 'Lifestyle_Score', 'No_Healthcare']


# 5. TRAIN / VALIDATION SPLIT

In [ ]:
# 5. TRAIN / VALIDATION SPLIT
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print(f"y_train distribution:\n{y_train.value_counts(normalize=True)}")

Train: (177112, 24), Val: (44278, 24)
y_train distribution:
History of HeartDisease or Attack
0.0    0.918391
1.0    0.081609
Name: proportion, dtype: float64


# 6. TRAIN Optuna

In [ ]:
# 6. TRAIN Optuna
def objective(trial):
    # Define integer versions of target locally
    y_train_int = y_train.astype(int)
    y_val_int = y_val.astype(int)

    params = {
        'n_estimators': 1000,
        'learning_rate': 0.03,
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 1.0),    # L1
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 1.0),  # L2
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'random_state': 42, 'n_jobs': -1, 'verbose': -1
    }

    m = lgb.LGBMClassifier(**params)
    m.fit(X_train, y_train_int)
    y_prob = m.predict_proba(X_val)[:, 1]

    _, best_f2 = max(
        [(t, fbeta_score(y_val_int, (y_prob >= t).astype(int), beta=2))
         for t in np.arange(0.1, 0.9, 0.01)],
        key=lambda x: x[1]
    )
    return best_f2

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"Best F2: {study.best_value:.4f}")
print(f"Best Params: {study.best_params}")

[I 2026-04-03 13:32:57,732] A new study created in memory with name: no-name-7af245c5-9af0-487d-82d5-6b95fbfa5314
[I 2026-04-03 13:33:21,727] Trial 0 finished with value: 0.5272259479522544 and parameters: {'max_depth': 9, 'num_leaves': 127, 'scale_pos_weight': 9.09392034173907, 'min_child_samples': 91, 'reg_alpha': 0.8724457829168852, 'reg_lambda': 0.7947178629320706, 'colsample_bytree': 0.8924788448319219, 'subsample': 0.9538269665632697}. Best is trial 0 with value: 0.5272259479522544.
[I 2026-04-03 13:33:38,051] Trial 1 finished with value: 0.5351353541903063 and parameters: {'max_depth': 5, 'num_leaves': 55, 'scale_pos_weight': 10.571101858890618, 'min_child_samples': 97, 'reg_alpha': 0.2100166183970485, 'reg_lambda': 0.17462187210682426, 'colsample_bytree': 0.7813544543085581, 'subsample': 0.8051987173375839}. Best is trial 1 with value: 0.5351353541903063.
[I 2026-04-03 13:33:54,134] Trial 2 finished with value: 0.5375440281780339 and parameters: {'max_depth': 4, 'num_leaves': 3

Best F2: 0.5397
Best Params: {'max_depth': 4, 'num_leaves': 59, 'scale_pos_weight': 10.469547875896062, 'min_child_samples': 82, 'reg_alpha': 0.32883272189248397, 'reg_lambda': 0.8129289224975669, 'colsample_bytree': 0.7396988777625271, 'subsample': 0.6638009344912061}


# CELL 7: FINAL MODEL

In [ ]:
best_params = {
    'max_depth': 4,
    'num_leaves': 59,
    'scale_pos_weight': 10.469547875896062,
    'min_child_samples': 82,
    'reg_alpha': 0.32883272189248397,
    'reg_lambda': 0.8129289224975669,
    'colsample_bytree': 0.7396988777625271,
    'subsample': 0.6638009344912061,
    'n_estimators': 1000,
    'learning_rate': 0.03,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

final_model = lgb.LGBMClassifier(**best_params)

# Train บน X ทั้งหมด (ไม่ใช่แค่ X_train) เพื่อให้โมเดลเห็นข้อมูลเต็ม
final_model.fit(X.astype(int), y.astype(int))

LGBMClassifier(colsample_bytree=0.7396988777625271, learning_rate=0.03,
               max_depth=4, min_child_samples=82, n_estimators=1000, n_jobs=-1,
               num_leaves=59, random_state=42, reg_alpha=0.32883272189248397,
               reg_lambda=0.8129289224975669,
               scale_pos_weight=10.469547875896062,
               subsample=0.6638009344912061, verbose=-1)

# 8. Find Best Threshold

In [ ]:
y_val_int = y_val.astype(int)
y_prob_val = final_model.predict_proba(X_val)[:, 1]
best_t, best_f2 = max(
    [(t, fbeta_score(y_val_int, (y_prob_val >= t).astype(int), beta=2))
     for t in np.arange(0.1, 0.9, 0.01)],
    key=lambda x: x[1]
)
print(f"Best Threshold : {best_t:.2f}")
print(f"Final F2 Score : {best_f2:.4f}")

Best Threshold : 0.56
Final F2 Score : 0.5502


# 9. PREDICT + SUBMIT

In [ ]:
test_prob = final_model.predict_proba(test_clean_fe)[:, 1]
test_pred = (test_prob >= 0.56).astype(int)

# เช็ค format ก่อน
print(sub_df.head())
print(sub_df.dtypes)


# Map 1/0 → Yes/No
submission = pd.DataFrame({
    'ID': ids.values,
    'History of HeartDisease or Attack': pd.Series(test_pred).map({1: 'Yes', 0: 'No'})
})

# เช็คก่อน save
print(submission.head())
print(f"\nYes: {(submission['History of HeartDisease or Attack'] == 'Yes').sum()}")
print(f"No : {(submission['History of HeartDisease or Attack'] == 'No').sum()}")
print(f"NaN: {submission['History of HeartDisease or Attack'].isna().sum()}")

submission.to_csv('submission.csv', index=False)
print("\nsubmission.csv saved!")

            ID History of HeartDisease or Attack
0  test_000001                                No
1  test_000002                                No
2  test_000003                                No
3  test_000004                               NaN
4  test_000005                               NaN
ID                                   object
History of HeartDisease or Attack    object
dtype: object
            ID History of HeartDisease or Attack
0  test_000001                                No
1  test_000002                                No
2  test_000003                               Yes
3  test_000004                                No
4  test_000005                                No

Yes: 21454
No : 52907
NaN: 0

submission.csv saved!
